## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
import keras
from  keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

I0000 00:00:1774169578.316010   65128 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774169581.407023   65128 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774169586.415618   65128 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [ ]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        #输入层
        self.W1 = tf.Variable(tf.random.normal([784, 256], stddev=0.1), name='w1')
        self.b1 = tf.Variable(tf.zeros([256]), name='b1')

        #输出层
        self.W2 = tf.Variable(tf.random.normal([256, 10], stddev=0.1), name='w2')
        self.b2 = tf.Variable(tf.zeros([10]), name='b2')
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x = tf.reshape(x, [-1, 784])

        #第一层
        h1 = tf.matmul(x, self.W1) + self.b1
        h1_relu = tf.nn.relu(h1)

        #第二层
        logits = tf.matmul(h1_relu, self.W2) + self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

E0000 00:00:1774169587.117968   65128 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

W0000 00:00:1774169588.591929   65128 cpu_allocator_impl.cc:82] Allocation of 188160000 exceeds 10% of free system memory.
W0000 00:00:1774169589.317665   65246 cpu_allocator_impl.cc:82] Allocation of 61440000 exceeds 10% of free system memory.
W0000 00:00:1774169590.000766   65246 cpu_allocator_impl.cc:82] Allocation of 61440000 exceeds 10% of free system memory.


epoch 0 : loss 2.8860307 ; accuracy 0.091466665


W0000 00:00:1774169592.390555   65128 cpu_allocator_impl.cc:82] Allocation of 188160000 exceeds 10% of free system memory.
W0000 00:00:1774169593.066993   65245 cpu_allocator_impl.cc:82] Allocation of 61440000 exceeds 10% of free system memory.


epoch 1 : loss 2.8209493 ; accuracy 0.092983335
epoch 2 : loss 2.7646523 ; accuracy 0.094733335
epoch 3 : loss 2.714929 ; accuracy 0.0952
epoch 4 : loss 2.6703389 ; accuracy 0.096316665
epoch 5 : loss 2.6298754 ; accuracy 0.097883336
epoch 6 : loss 2.5927873 ; accuracy 0.0989
epoch 7 : loss 2.5584939 ; accuracy 0.10016666
epoch 8 : loss 2.5265381 ; accuracy 0.1022
epoch 9 : loss 2.496556 ; accuracy 0.103933334
epoch 10 : loss 2.4682481 ; accuracy 0.106466666
epoch 11 : loss 2.4413717 ; accuracy 0.109283336
epoch 12 : loss 2.4157295 ; accuracy 0.11271667
epoch 13 : loss 2.391154 ; accuracy 0.116333336
epoch 14 : loss 2.3675098 ; accuracy 0.12055
epoch 15 : loss 2.3446817 ; accuracy 0.12518333
epoch 16 : loss 2.322577 ; accuracy 0.13013333
epoch 17 : loss 2.3011177 ; accuracy 0.13556667
epoch 18 : loss 2.2802358 ; accuracy 0.1416
epoch 19 : loss 2.259877 ; accuracy 0.14785
epoch 20 : loss 2.2399943 ; accuracy 0.15453333
epoch 21 : loss 2.2205484 ; accuracy 0.162
epoch 22 : loss 2.2015078